# Combined Feature Extraction Pipeline (All Families)

Builds one merged dataset from all configured feature families and can auto-run missing family notebooks.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


REPO_ROOT = find_repo_root(Path.cwd())
PIPELINE_DIR = REPO_ROOT / 'feature_extraction' / 'pipelines'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from pipeline_common import (
    ANOVA_FAMILIES,
    DEFAULT_ALL_FAMILIES,
    METADATA_COLUMNS,
    feature_columns_from_families,
    load_base_metadata,
    load_family_frames,
    load_selected_feature_lists,
    merge_feature_families,
    run_missing_family_generators,
)

OUT_DIR = REPO_ROOT / 'extracted_features' / 'combined'
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
FAMILIES = DEFAULT_ALL_FAMILIES.copy()
AUTO_RUN_MISSING = True
NOTEBOOK_TIMEOUT_SECONDS = 7200  # fail fast instead of hanging forever

if AUTO_RUN_MISSING:
    gen_results = run_missing_family_generators(
        REPO_ROOT,
        families=FAMILIES,
        execute_timeout=NOTEBOOK_TIMEOUT_SECONDS,
    )
    if gen_results:
        display(pd.DataFrame(gen_results))


In [ ]:
base_df = load_base_metadata(REPO_ROOT, include_xxx=False, require_agreement=True)
family_frames, family_report = load_family_frames(
    REPO_ROOT,
    families=FAMILIES,
    prefix_features=True,
)

report_df = pd.DataFrame(family_report).sort_values(['available', 'family'], ascending=[False, True])
display(report_df)

merged_df = merge_feature_families(base_df, family_frames)
feature_cols = feature_columns_from_families(merged_df, list(family_frames.keys()))
all_nan_cols = [c for c in feature_cols if merged_df[c].isna().all()]
if all_nan_cols:
    merged_df = merged_df.drop(columns=all_nan_cols)
    feature_cols = [c for c in feature_cols if c not in all_nan_cols]

meta_cols = [c for c in METADATA_COLUMNS if c in merged_df.columns]
combined_df = merged_df[[*meta_cols, *feature_cols]].copy()

print(f'Base rows: {len(base_df):,}')
print(f'Loaded families: {len(family_frames):,} / {len(FAMILIES):,}')
print(f'Combined shape: {combined_df.shape}')
combined_df.head(2)


In [ ]:
OUT_CSV = OUT_DIR / 'all_families_features.csv'
combined_df.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')
